In [1]:
!pip install -q transformers torch accelerate sentencepiece pandas tqdm

In [2]:
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#Cek GPU

import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


Load model dari gdrive

In [5]:
MODEL_PATH = "/content/drive/MyDrive/SKRIPSI MANTAP/indobertweet"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModel.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

print("Model berhasil dimuat")
print("Device :", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: /content/drive/MyDrive/SKRIPSI MANTAP/indobertweet
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model berhasil dimuat
Device : cuda


In [6]:
# Path folder di Google Drive
folder_path = '/content/drive/MyDrive/SKRIPSI MANTAP/data sosmed/.Process/Clean data'

# Path file
file_path = f'{folder_path}/tt_clean.csv'

# Membaca CSV
df = pd.read_csv(
    file_path,
    sep=',',  # gunakan ',' jika disimpan dengan to_csv() default
    engine='python'
)

# Cek nama kolom
print(df.columns)

# Ambil kolom comment
texts = df["tt_clean"]

print("Jumlah komentar:", len(texts))

Index(['tt_clean'], dtype='object')
Jumlah komentar: 16339


In [11]:
#parameter
MAX_LENGTH = 50
BATCH_SIZE = 32

In [12]:
def extract_embeddings(texts):

    all_embeddings = []

    texts = texts.fillna("").astype(str).reset_index(drop=True)

    for i in tqdm(range(0, len(texts), BATCH_SIZE)):

        batch = texts.iloc[i:i+BATCH_SIZE].astype(str).tolist()

        encoded = tokenizer(
            batch,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        encoded = {
            k: v.to(device)
            for k, v in encoded.items()
        }

        with torch.no_grad():
            outputs = model(**encoded)

        embeddings = outputs.last_hidden_state.cpu().numpy()

        all_embeddings.append(embeddings)

    return np.concatenate(all_embeddings, axis=0)

In [9]:
X_embedding = extract_embeddings(df["tt_clean"])

  0%|          | 0/511 [00:00<?, ?it/s]

In [10]:
print(X_embedding.shape)

(16339, 50, 768)


In [ ]:
np.save(
    "/content/drive/MyDrive/SKRIPSI MANTAP/data sosmed/Tiktok/tt_embed_bilstm.npy",
    X_embedding
)